# LLM Lab — Fine-Tuning — AI Lab Instructor Notebook

*NLP & Transformers · Hard*

Simulate fine-tuning with a tiny NumPy MLP: pretrain on a larger source task, transfer the learned feature extractor to a smaller target task, and compare scratch vs frozen features vs full fine-tuning.

**Outcome.** By the end, you will understand how pretraining, feature reuse, freezing, full fine-tuning, and feature drift relate to the broader LLM fine-tuning workflow.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# LLM Lab — Fine-Tuning (Offline, NumPy)

We simulate fine-tuning dynamics with a tiny MLP: pretrain on Task A, then adapt to Task B.

This is **not** a real LLM fine-tune. It is a compact offline lab for understanding the workflow:
- reuse a pretrained representation
- freeze vs unfreeze the feature extractor
- compare transfer to training from scratch
- inspect sample efficiency and feature drift

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** This setup cell only defines the shared NumPy environment, deterministic RNG, and the `check(...)` helper used across the rest of the lab.

## Task 2: Section 0 — Data: Task A (pretrain) and Task B (fine-tune)
*Section: Data: Task A and Task B*

## Section 0 — Data: Task A (pretrain) and Task B (fine-tune)

Create two synthetic supervised tasks:
- **Task A**: the larger source task used for pretraining
- **Task B**: the smaller target task used for fine-tuning

The lab uses deterministic train/validation splits so strategy comparisons are fair.

In [ ]:
$f

In [ ]:
# Checks
check('labelsA', int(yA.min()) == 0 and int(yA.max()) == kA-1)
check('labelsB', int(yB.min()) == 0 and int(yB.max()) == kB-1)
check('split_sizes', XA_tr.shape[0] == 4800 and XA_va.shape[0] == 1200 and XB_tr.shape[0] == 480 and XB_va.shape[0] == 120)


**Why this works.** The source task is intentionally larger than the target task so transfer learning has a reason to help. The validation split is deterministic so scratch, frozen-transfer, and full fine-tune are directly comparable.

## Task 3: Section 1 — Model: tiny MLP = feature extractor + head
*Section: Model*

## Section 1 — Model: tiny MLP = feature extractor + head

Implement a tiny two-part model:
- **feature extractor**: `W1, b1`
- **task head**: `W2, b2`

That separation is what makes freezing vs full fine-tuning meaningful.

In [ ]:
$10

In [ ]:
# Checks
params_tmp = init_mlp(d_in=d, d_hidden=32, k_out=kB, seed=0)
H_tmp, cache_tmp = forward_feats(params_tmp, XB_tr[:5])
logits_tmp = forward_logits(params_tmp, H_tmp)
_, dlogits_tmp = xent_loss_and_grad_logits(logits_tmp, yB_tr[:5])
grads_frozen = backward(params_tmp, cache_tmp, dlogits_tmp, freeze_feats=True)
check('feat_shape', H_tmp.shape == (5, 32))
check('logits_shape', logits_tmp.shape == (5, kB))
check('freeze_zeroes_feature_grads', np.allclose(grads_frozen['W1'], 0.0) and np.allclose(grads_frozen['b1'], 0.0))


**Why this works.** This lab models fine-tuning as "reuse the hidden representation, replace or adapt the head." Freezing works by explicitly zeroing feature-extractor gradients while still updating the output head.

## Task 4: Section 2 — Training utilities
*Section: Training utilities*

## Section 2 — Training utilities

Write the helpers that make the transfer comparison possible:
- minibatch iteration
- one training epoch
- evaluation on a held-out split

These utilities should work for both pretraining and fine-tuning.

In [ ]:
def minibatches(X, y, batch_size: int, seed: int):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    perm = rng.permutation(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        yield X[idx], y[idx]

def train_epoch(params, X, y, lr: float, batch_size: int, seed: int, freeze_feats: bool):
    losses = []
    accs = []
    for Xb, yb in minibatches(X, y, batch_size=batch_size, seed=seed):
        H, cache = forward_feats(params, Xb)
        logits = forward_logits(params, H)
        loss, dlogits = xent_loss_and_grad_logits(logits, yb)
        grads = backward(params, cache, dlogits, freeze_feats=freeze_feats)
        sgd_step(params, grads, lr=lr)
        losses.append(loss)
        accs.append(accuracy(logits, yb))
    return float(np.mean(losses)), float(np.mean(accs))

def eval_model(params, X, y):
    H, _ = forward_feats(params, X)
    logits = forward_logits(params, H)
    loss, _ = xent_loss_and_grad_logits(logits, y)
    return loss, accuracy(logits, y)


In [ ]:
# Checks
params_test = init_mlp(d_in=d, d_hidden=24, k_out=kB, seed=3)
tr_loss, tr_acc = train_epoch(params_test, XB_tr[:128], yB_tr[:128], lr=0.1, batch_size=32, seed=0, freeze_feats=False)
va_loss, va_acc = eval_model(params_test, XB_va[:64], yB_va[:64])
check('epoch_metrics_finite', np.isfinite(tr_loss) and np.isfinite(tr_acc) and tr_loss > 0.0)
check('eval_acc_range', np.isfinite(va_loss) and 0.0 <= va_acc <= 1.0)


**Why this works.** The same training loop is reused in three regimes: scratch, frozen-transfer, and full fine-tune. That keeps the strategy comparison clean because only initialization and the `freeze_feats` flag change.

## Task 5: Section 3 — Pretrain on Task A
*Section: Pretrain*

## Section 3 — Pretrain on Task A

Train the full model on the larger source task and then save the learned feature extractor weights.

This simulates pretraining: the hidden layer learns reusable structure before we ever touch the target task.

In [ ]:
d_hidden = 96
params_A = init_mlp(d_in=d, d_hidden=d_hidden, k_out=kA, seed=0)
pre_losses = []
for epoch in range(12):
    tr_loss, tr_acc = train_epoch(params_A, XA_tr, yA_tr, lr=0.2, batch_size=256, seed=epoch, freeze_feats=False)
    va_loss, va_acc = eval_model(params_A, XA_va, yA_va)
    pre_losses.append((tr_loss, va_loss, va_acc))
    print('epoch', epoch+1, 'tr_loss', round(tr_loss,4), 'va_loss', round(va_loss,4), 'va_acc', round(va_acc,3))
check('pretrain_runs', np.isfinite(pre_losses[-1][1]))

# Save a copy of pretrained feature extractor
W1_pre = params_A['W1'].copy()
b1_pre = params_A['b1'].copy()


In [ ]:
# Checks
check('pretrain_runs', np.isfinite(pre_losses[-1][1]))
check('pretrained_features_saved', W1_pre.shape == params_A['W1'].shape and b1_pre.shape == params_A['b1'].shape)


**Why this works.** Pretraining is just ordinary supervised learning on Task A, but we keep the hidden-layer weights because those are the reusable representation we want to transfer into Task B.

## Task 6: Section 4 — Fine-tune on Task B (compare strategies)
*Section: Fine-tune*

## Section 4 — Fine-tune on Task B (compare strategies)

Compare three target-task strategies:
- **scratch**: random initialization on Task B
- **freeze**: initialize from pretraining, train a new head only
- **full**: initialize from pretraining, then fine-tune all parameters

Also measure **feature drift** so you can see whether full fine-tuning actually changes the representation.

In [ ]:
$12

In [ ]:
# Checks
check('ft_lists', len(scratch_hist) == 15 and len(freeze_hist) == 15 and len(full_hist) == 15)
check('full_beats_freeze', full_hist[-1] >= freeze_hist[-1])
check('drift_positive', np.isfinite(drift) and drift > 0.0)


**Why this works.** This section is the real point of the lab. The three strategies use the same training loop but different initialization and freezing choices. That isolates the practical fine-tuning tradeoff: reusing features helps only if the source representation is relevant and the adaptation strategy matches the data regime.